In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src import nulls
from src.notebook import load_notebook_data
from src.timescales import (convert_data_dict, average_data_dict_converted,
                             compute_regional_corrs, compute_deflections,
                             create_ints_dataframe_custom_grouping)
from src.gams import apply_region_mask
from snaplab_tools.plotting.plotting import plot_correlation

In [ ]:
dataset = 'HCP-YA'
which_modality = 'FMRI'
use_7t = False
# output_type = 'timescale_zero_crossing'
output_type = 'timescale_auc_zero_crossing'

# output_type = output_type + '_lp-0.01hz'
# output_type = output_type + '_hp-0.01hz'
# output_type = output_type + '_bp-0.01-0.08hz'
# output_type = output_type + '_bp-0.08-0.15hz'

use_resids = True
notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   use_7t=use_7t, output_type=output_type, use_resids=use_resids,
                                   load_subject_data=False)

In [ ]:
nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
# get key outputs for this notebook
timescales_processor = notebook_data['timescales_processor']
data_dict = notebook_data['data_dict']
data_dict_tsnr = notebook_data['data_dict_tsnr']
results_dir = notebook_data['results_dir']

df_neuro = notebook_data['supporting_data']['df_neuro']
yeo_systems = notebook_data['supporting_data']['yeo_systems']

print(data_dict.keys())
print(df_neuro.columns)

In [ ]:
n_regions = len(yeo_systems)
n_systems = len(np.unique(yeo_systems))

In [ ]:
subject_ids = list(dict.fromkeys(scan.subject_id for scan in timescales_processor.scan_registry))
n_subs = len(subject_ids)
print(f'Number of subjects: {n_subs}')

In [ ]:
timescales_processor.scan_registry[:5]

In [ ]:
if dataset == 'HCP-YA':
    scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST1RL', 'rfMRIREST2LR', 'rfMRIREST2RL']
    # scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST1RL']
    # scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST2LR']
    condition_name = 'WM' # 'ALL', 'WM', 'EMOTION', 'GAMBLING', 'LANGUAGE', 'RELATIONAL', 'SOCIAL'

    if 'ALL' in condition_name:
        tasks = [
                'tfMRIWMLR', 'tfMRIWMRL',
                'tfMRIEMOTIONLR', 'tfMRIEMOTIONRL',
                'tfMRIGAMBLINGLR', 'tfMRIGAMBLINGRL',
                'tfMRILANGUAGELR', 'tfMRILANGUAGERL',
                'tfMRIRELATIONALLR', 'tfMRIRELATIONALRL',
                'tfMRISOCIALLR', 'tfMRISOCIALRL',
                ]
        scans_to_keep = scans_to_keep + tasks
        task_performance = []
    elif 'WM' in condition_name:
        scans_to_keep.append('tfMRIWMLR')
        scans_to_keep.append('tfMRIWMRL')
        task_performance = ['WM_Task_Acc', 'WM_Task_Median_RT']
    elif 'EMOTION' in condition_name:
        scans_to_keep.append('tfMRIEMOTIONLR')
        scans_to_keep.append('tfMRIEMOTIONRL')
        task_performance = ['EMOTION_Task_Acc', 'EMOTION_Task_Median_RT']
    elif 'GAMBLING' in condition_name:
        scans_to_keep.append('tfMRIGAMBLINGLR')
        scans_to_keep.append('tfMRIGAMBLINGRL')
        task_performance = ['GAMBLING_Task_Perc_Larger', 'GAMBLING_Task_Perc_Smaller']
    elif 'LANGUAGE' in condition_name:
        scans_to_keep.append('tfMRILANGUAGELR')
        scans_to_keep.append('tfMRILANGUAGERL')
        task_performance = ['LANGUAGE_Task_Acc', 'LANGUAGE_Task_Median_RT']
    elif 'RELATIONAL' in condition_name:
        scans_to_keep.append('tfMRIRELATIONALLR')
        scans_to_keep.append('tfMRIRELATIONALRL')
        task_performance = ['RELATIONAL_Task_Acc', 'RELATIONAL_Task_Median_RT']
    elif 'SOCIAL' in condition_name:
        scans_to_keep.append('tfMRISOCIALLR')
        scans_to_keep.append('tfMRISOCIALRL')
        task_performance = ['SOCIAL_Task_Perc_Random', 'SOCIAL_Task_Perc_TOM'] 

    # scan_registry_filtered = pd.DataFrame(columns=timescales_processor.scan_registry.columns)
    # for scan in scans_to_keep:
    #     # print(scan)
    #     subject_filter = timescales_processor.scan_registry['task'] == scan
    #     scan_registry_filtered = pd.concat((scan_registry_filtered, timescales_processor.scan_registry.loc[subject_filter])).reset_index(drop=True)
        
    # timescales_processor.scan_registry = scan_registry_filtered

In [ ]:
timescales_processor.match_registry_to_behavior()
print(timescales_processor.df_behavior_matched.shape)

In [ ]:
df_behavior = timescales_processor.df_behavior_matched.copy()
df_behavior.head()

# Data for figures

In [ ]:
brain_map = 'SA_Axis'

data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_converted = average_data_dict_converted(data_dict_converted, average_3t_rest=True, average_3t_task_pairs=True)
data_dict_converted.keys()

## Subject-level analyses: regional

## INT deflection variance and covariance

In [ ]:
# ── Deflection method toggle ──────────────────────────────────────────────────
# 'simple' : |task - rest|  (compute_deflections, nuisance_regression=False)
# 'glm'    : task residual after OLS(task ~ rest)  (compute_deflections, nuisance_regression=True)
#            For regional behaviour correlations, compute_deflections_glm is used instead of
#            computing deflections then running spearmanr.
deflection_method = 'simple'   # 'simple' | 'glm' | 'matrix'
# ─────────────────────────────────────────────────────────────────────────────

# Neural data for rest and task
rest_data = data_dict_converted['task-REST_3T']           # (n_subjects, n_regions)
task_data = data_dict_converted[f'task-{condition_name}'] # (n_subjects, n_regions)

behavior_measures = [
    f'{condition_name}_Task_Acc',
    f'{condition_name}_Task_Median_RT',
    'CogTotalComp_Unadj',
]

# _use_nuisance = (deflection_method == 'glm')
_use_nuisance = True
# _use_nuisance = False

# Compute subject-level absolute deflection matrix (n_subjects × n_regions)
deflection_matrix = np.column_stack([
    compute_deflections(rest_data[:, r], task_data[:, r],
                        nuisance_regression=_use_nuisance, absolute=True)
    for r in range(n_regions)
])

# Build tSNR mask: exclude lowest 10% tSNR regions for the task condition
df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr, average_3t_task_pairs=True)
tsnr_col = f'task-{condition_name}'
region_mask = (df_tsnr[tsnr_col] < np.quantile(df_tsnr[tsnr_col], q=0.1)).values

# Apply mask — drops low-tSNR regions from data arrays and SA axis
sa_axis_masked = df_neuro[brain_map].values.copy()
rest_data, task_data, deflection_matrix, sa_axis_masked = apply_region_mask(
    region_mask, rest_data, task_data, deflection_matrix, sa_axis=sa_axis_masked)
yeo_systems_masked = np.asarray(yeo_systems)[~region_mask]

# Per-region statistics on masked data
mean_deflection = np.nanmean(deflection_matrix, axis=0)  # (n_regions_valid,)
std_deflection  = np.nanstd(deflection_matrix, axis=0)   # (n_regions_valid,)

print(f'deflection_method  : {deflection_method}')
print(f'deflection_matrix  : {deflection_matrix.shape}')
print(f'mean_deflection    : {mean_deflection.shape}')


In [ ]:
# ── Split-half reliability data ───────────────────────────────────────────────
_data_raw = convert_data_dict(data_dict, timescales_processor.scan_registry, verbose=False)

def _stack_mean(d, patterns):
    arrays = [d[k] for k in d if any(p in k for p in patterns) and d[k] is not None]
    return np.nanmean(np.stack(arrays, axis=0), axis=0) if arrays else None

_rest_lr = _stack_mean(_data_raw, ['rfMRIREST1LR', 'rfMRIREST2LR'])
_rest_rl = _stack_mean(_data_raw, ['rfMRIREST1RL', 'rfMRIREST2RL'])
_task_lr = _stack_mean(_data_raw, ['tfMRIWMLR'])
_task_rl = _stack_mean(_data_raw, ['tfMRIWMRL'])

_rest_lr_m = _rest_lr[:, ~region_mask]
_rest_rl_m = _rest_rl[:, ~region_mask]
_task_lr_m = _task_lr[:, ~region_mask]
_task_rl_m = _task_rl[:, ~region_mask]

def _cv_defl(a, b):
    defl = np.abs(b - a)
    return np.nanstd(defl, axis=0) / np.nanmean(defl, axis=0)

cv_defl_lr = _cv_defl(_rest_lr_m, _task_lr_m)
cv_defl_rl = _cv_defl(_rest_rl_m, _task_rl_m)

print(f'Split-half r: {float(sp.stats.pearsonr(cv_defl_lr, cv_defl_rl)[0]):.3f}')


In [ ]:
# ── Spatial nulls for the S-A-axis correlations ───────────────────────────────
# Per-hemisphere geodesic BrainSMASH surrogates of the S-A axis (src.nulls; shared with fig3/fig5).
# Each S-A panel below is tested by correlating its (INT-derived) map against these S-A surrogates,
# i.e. a spatial-autocorrelation-preserving null. Surrogates cache under results/<dataset>/brainsmash_nulls.
N_PERMS        = 5000
n_regions_full = len(region_mask)   # 400 (pre-mask); masked maps are re-expanded to this length

distance_matrix, hemi_labels = nulls.load_distance_matrix(n_regions_full, kind='geodesic')
surrogate_cache_dir = nulls.surrogate_cache_dir(timescales_processor, dataset)

sa_axis_full = df_neuro[brain_map].values   # dense S-A axis over all 400 parcels
surr_sa = nulls.generate_surrogates(
    sa_axis_full, name='SA_Axis', n_regions=n_regions_full, n_perms=N_PERMS, kind='geodesic',
    distance_matrix=distance_matrix, hemi=hemi_labels, cache_dir=surrogate_cache_dir)

def _to_full(vec_masked):
    """Re-expand a masked (valid-region) vector to the full 400-parcel grid (NaN at masked regions)."""
    full = np.full(n_regions_full, np.nan)
    full[~region_mask] = vec_masked
    return full

print(f'S-A surrogates: {surr_sa.shape}  (n_perms x n_regions)')

# Figure 2

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=len(behavior_measures),
                         sharex=False, sharey=False,
                         figsize=(len(behavior_measures)*2.1, 4), dpi=300)

sa_null_rows = []   # (panel, r, p_smash) for the S-A-axis correlations -> saved to results_dir

# Top row: variance characterization (no behavior), tested against the geodesic S-A null
variance_panels = [
    (mean_deflection, '|INT Deflections| (Mean)'),
    (std_deflection,  '|INT Deflections| (Std. Dev.)'),
]
for i, (stat_data, stat_label) in enumerate(variance_panels):
    d = nulls.corr_with_null(_to_full(stat_data), sa_axis_full, surr_sa, method='pearson')
    sa_null_rows.append({'panel': stat_label, 'r': d['r'], 'p_smash': d['p_smash']})
    plot_correlation(sa_axis_masked, stat_data, axes[0, i],
                     data_group=yeo_systems_masked, show_stats=True, fontsize=8, method='pearson',
                     y_label=stat_label, x_label=' ', size=10, cbar_label='Yeo Systems',
                     show_colorbar=False, data_group_cmap='yeo7',
                     auto_polynomial=False, custom_pvalue=d['p_smash'])

# Top row panel 3: split-half reliability (CV LR vs RL) — parametric (not a spatial-null test)
plot_correlation(cv_defl_lr, cv_defl_rl, axes[0, 2],
                 data_group=yeo_systems_masked, show_stats=True, fontsize=8, method='pearson',
                 y_label='CV |INT Deflection| (RL)', x_label='CV |INT Deflection| (LR)',
                 size=10, cbar_label='Yeo Systems', show_colorbar=False,
                 data_group_cmap='yeo7', auto_polynomial=False)
# axes[0, 2].set_title('Phase-encoding consistency', fontsize=8, pad=3)

# Bottom row: behavioral covariance, tested against the geodesic S-A null
for i, behavior_measure in enumerate(behavior_measures):
    behavior_data = df_behavior[behavior_measure].values.astype(float)
    rest_correlations, task_correlations, deflection_correlations = compute_regional_corrs(
        rest_data, task_data, behavior_data, nuisance_regression=_use_nuisance, deflection_method=deflection_method)

    if 'Acc' in behavior_measure:
        y_sublabel = 'Acc.'
    elif 'RT' in behavior_measure:
        y_sublabel = 'RT'
    else:
        y_sublabel = 'Cognition'

    file_label = f'Corr(INT Deflections, {y_sublabel})'
    d = nulls.corr_with_null(_to_full(deflection_correlations), sa_axis_full, surr_sa, method='pearson')
    sa_null_rows.append({'panel': file_label, 'r': d['r'], 'p_smash': d['p_smash']})
    plot_correlation(sa_axis_masked, deflection_correlations, axes[1, i],
                     data_group=yeo_systems_masked, show_stats=True, fontsize=8, method='pearson',
                     y_label=file_label, x_label='S-A axis', size=10, cbar_label='Yeo Systems',
                     show_colorbar=False, data_group_cmap='yeo7',
                     auto_polynomial=False, custom_pvalue=d['p_smash'])

for ax in axes.flat:
    ax.yaxis.set_label_coords(-0.3, 0.5)
    ax.tick_params(axis='y', which='major', pad=1)
    ax.tick_params(axis='x', which='major', pad=1)
    ax.xaxis.labelpad = 2

fig.tight_layout()
fig.savefig(os.path.join(nas_scratch_dir, f'corr_ints_sa.svg'), bbox_inches='tight', dpi=1000, transparent=True)
plt.show()

# S-A-axis spatial-null statistics (goal 6: results -> results_dir)
pd.DataFrame(sa_null_rows).to_csv(os.path.join(results_dir, 'corr_ints_sa_nulls.csv'), index=False)

In [ ]:
# another one just for the colorbar
fig, axes = plt.subplots(nrows=1, ncols=1,
                         sharex=True, sharey=False,
                         figsize=(2, 2), dpi=300)

plot_correlation(sa_axis_masked, mean_deflection, axes, 
                data_group=yeo_systems_masked, show_stats=True, fontsize=8,
                y_label=file_label, x_label='S-A axis', size=10, cbar_label='Yeo Systems', show_colorbar=True, data_group_cmap='yeo7',
                auto_polynomial=True, models_to_test=[1, 2], model_selection_metric='variance_explained')

fig.savefig(os.path.join(nas_scratch_dir, f'corr_ints_sa_colorbar.png'), bbox_inches='tight', dpi=1000, transparent=True)
plt.show()